In [ ]:
import os
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append("../src")
import config

def read_json(name):
    with open(os.path.join(config.REPORTS_DIR, name)) as f:
        return json.load(f)

detector = read_json("detector_metrics.json")
baselines = read_json("baseline_metrics.json")

In [ ]:
rows = []
for name in baselines:
    rows.append({
        "model": name,
        "PR-AUC": baselines[name]["pr_auc"],
        "ROC-AUC": baselines[name]["roc_auc"],
        "Brier": baselines[name]["brier"],
        "recall at 0.5": baselines[name]["at_threshold"]["recall"],
        "precision at 0.5": baselines[name]["at_threshold"]["precision"],
    })
rows.append({
    "model": "xgboost",
    "PR-AUC": detector["pr_auc"],
    "ROC-AUC": detector["roc_auc"],
    "Brier": detector["brier"],
    "recall at 0.5": detector["at_threshold"]["recall"],
    "precision at 0.5": detector["at_threshold"]["precision"],
})
pd.DataFrame(rows)

In [ ]:
curve = pd.read_csv(os.path.join(config.REPORTS_DIR, "pr_curve_val.csv"))
plt.figure(figsize=(6, 5))
plt.plot(curve["recall"], curve["precision"], color="#2F4B7C")
plt.xlabel("recall")
plt.ylabel("precision")
plt.title("XGBoost precision-recall curve (validation)")
plt.xlim(0, 1.0)
plt.ylim(0, 1.02)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
per_class = pd.read_csv(os.path.join(config.REPORTS_DIR, "detector_per_class_recall.csv"))
per_class = per_class.sort_values("rate_at_0.5")
colors = []
for kind in per_class["kind"]:
    if kind == "specificity":
        colors.append("#999999")
    else:
        colors.append("#4C72B0")
plt.figure(figsize=(8, 5))
plt.barh(per_class["label"], per_class["rate_at_0.5"], color=colors)
plt.xlabel("recall (attack classes) or specificity (BENIGN) at threshold 0.5")
plt.xlim(0, 1.02)
plt.title("Per-class rate on validation")
plt.tight_layout()
plt.show()
per_class

In [ ]:
importance = pd.read_csv(os.path.join(config.REPORTS_DIR, "feature_importance.csv"))
top = importance.head(20).iloc[::-1]
plt.figure(figsize=(8, 6))
plt.barh(top["feature"], top["share"], color="#4C72B0")
plt.xlabel("share of total gain")
plt.title("Top 20 features by gain")
plt.tight_layout()
plt.show()
print("features with zero gain:", int((importance["gain"] == 0).sum()))

In [ ]:
ablation_path = os.path.join(config.REPORTS_DIR, "dest_port_ablation.md")
if os.path.exists(ablation_path):
    print(open(ablation_path, encoding="utf-8").read())
else:
    print("Run train_detector.py --no-dest-port to create the ablation report.")